# 💬 BitNet b1.58 Chat & Inference Console
### Interactive Jupyter Notebook to load `bitnet_model_ternary.pt` and chat with the 350M parameters ternary model.

This notebook allows you to load the packed 2-bit weights generated from the conversion pipeline, unpack them into ternary values $\{-1, 0, 1\}$, load them in a clean model skeleton (rebuilt directly from the architecture config, without downloading the original weights of Facebook OPT), and open an interactive chat interface inside your environment. It calculates the generation speed in **tokens per second** and lets you adjust inference parameters like `top_k`, `temperature`, and `max_new_tokens` on the fly.

# 🛠️ Step 1: Environment Setup & Helper Definitions
First, we import PyTorch and define the custom `BitLinear` layers, the weight unpacking logic, and memory clearing utilities.

In [ ]:
# ==============================================================================
# STEP 1: INITIAL COMPONENT DEFINITIONS
# ==============================================================================
import os
import sys
import time
import gc
import ctypes
import torch
import torch.nn as nn
import torch.nn.functional as F

# Check CUDA device availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Execution Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU Name: {torch.cuda.get_device_name(0)}")

def clean_system_ram():
    """Aggressively releases unused RAM from both GPU cache and system heap back to OS."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    try:
        libc = ctypes.CDLL("libc.so.6")
        libc.malloc_trim(0)
    except Exception:
        pass

def quantize_activation(x):
    """Quantize activations to 8-bit per token."""
    scale = 127.0 / (torch.max(torch.abs(x), dim=-1, keepdim=True).values + 1e-5)
    x_quant = torch.clamp(torch.round(x * scale), -128, 127)
    x_dequant = x_quant / scale
    return x + (x_dequant - x).detach()

def unpack_ternary_weights(packed, original_shape):
    """Unpack uint8 byte array back into ternary {-1,0,1} tensor."""
    val0 = packed & 3
    val1 = (packed >> 2) & 3
    val2 = (packed >> 4) & 3
    val3 = (packed >> 6) & 3
    flat = torch.stack([val0, val1, val2, val3], dim=1).view(-1)
    flat_signed = torch.where(flat == 2, torch.tensor(-1, dtype=torch.int8, device=packed.device), flat.to(torch.int8))
    num_elements = original_shape[0] * original_shape[1]
    return flat_signed[:num_elements].view(original_shape)

class BitLinear(nn.Module):
    """
    BitLinear Layer with Grouped Ternary Weight Quantization {-1, 0, 1}
    and 8-bit Activation Quantization.
    """
    def __init__(self, in_features, out_features, bias=True, group_size=128):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.group_size = group_size
        self.weight = nn.Parameter(torch.zeros(out_features, in_features))
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_features))
        else:
            self.register_parameter('bias', None)
        self.ternary_mode = True  # Loaded models will execute in pure inference ternary mode
            
    def forward(self, x):
        w = self.weight.to(x.dtype)
        bias = self.bias.to(x.dtype) if self.bias is not None else None
        x_quant = quantize_activation(x)
        return F.linear(x_quant, w, bias)
        
    def load_ternary_weights(self, w_quant, scale_factors):
        """Dequantize ternary weights using scale factors and copy in place."""
        out_features, in_features = w_quant.shape
        num_groups = scale_factors.shape[1]
        scales_expanded = scale_factors.unsqueeze(-1).expand(out_features, num_groups, self.group_size)
        scales_expanded = scales_expanded.reshape(out_features, -1)[:, :in_features]
        dequantized_weight = w_quant.to(scale_factors.dtype) * scales_expanded
        self.weight.data.copy_(dequantized_weight)
        self.ternary_mode = True

def convert_model_to_bitnet(model, group_size=128):
    """Replaces nn.Linear layers with BitLinear framework layers."""
    def replace_linears(module):
        for name, child in list(module.named_children()):
            if isinstance(child, nn.Linear):
                if name == "lm_head":
                    continue
                bit_linear = BitLinear(
                    in_features=child.in_features,
                    out_features=child.out_features,
                    bias=child.bias is not None,
                    group_size=group_size
                )
                bit_linear.to(child.weight.dtype)
                setattr(module, name, bit_linear)
            else:
                replace_linears(child)

    if hasattr(model, "model") and hasattr(model.model, "decoder"):
        replace_linears(model.model.decoder.layers)
    else:
        replace_linears(model)
    return model

: 

# 📥 Step 2: Load the Model Configuration and Unpack Ternary Weights
We load the configuration and tokenizer from `facebook/opt-350m` (only metadata is downloaded, not the weights). We build a clean model skeleton with randomized weights and overwrite them with our local `bitnet_model_ternary.pt`.

In [ ]:
# ==============================================================================
# STEP 2: LOAD TERNARY WEIGHTS FROM DISK
# ==============================================================================
try:
    import transformers
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "transformers", "accelerate"])

from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM

MODEL_ID = "facebook/opt-350m"
WEIGHTS_PATH = "bitnet_model_ternary.pt"

if not os.path.exists(WEIGHTS_PATH):
    raise FileNotFoundError(f"❌ '{WEIGHTS_PATH}' not found in current directory! Please convert and export the weights first.")

print("📥 Loading tokenizer and config (small files).../n")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
config = AutoConfig.from_pretrained(MODEL_ID)

print("🏗️ Rebuilding empty model skeleton from configuration (no weights downloaded!)...")
# Initialize the model skeleton with config (creates random tensors to be overwritten)
model = AutoModelForCausalLM.from_config(config).to(device)
model = convert_model_to_bitnet(model, group_size=128)

print("🔓 Loading and unpacking local ternary weights directly into model...")
loaded_dict = torch.load(WEIGHTS_PATH, map_location=device)

for name, module in model.named_modules():
    if isinstance(module, BitLinear):
        packed_key = f"{name}.weight_ternary_packed"
        if packed_key in loaded_dict:
            # Unpack the weights
            packed_w = loaded_dict[packed_key].to(device)
            shape = tuple(loaded_dict[f"{name}.original_shape"].tolist())
            w_quant = unpack_ternary_weights(packed_w, shape).to(device)
            scale_factors = loaded_dict[f"{name}.scale_factors"].to(device)
            
            # Load dequantized weight matrix in-place
            module.load_ternary_weights(w_quant, scale_factors)
            
            if f"{name}.bias" in loaded_dict:
                module.bias.data.copy_(loaded_dict[f"{name}.bias"])
    else:
        # Load standard weights for normal Layers (lm_head, embeddings)
        for param_name, param in module.named_parameters(recurse=False):
            full_name = f"{name}.{param_name}" if name else param_name
            if full_name in loaded_dict:
                param.data.copy_(loaded_dict[full_name])

model.eval()        # Switch to eval mode

del loaded_dict     # Release dictionary reference from CPU RAM
clean_system_ram()
print("\n✅ Pure Ternary BitNet model loaded and ready!")

# ⚙️ Step 3: Interactive Configuration Form
Use the sliding controls on the right panel in Google Colab to modify the generation parameters. Adjusting these values will affect the chat loop in real-time.

In [ ]:
#@title ⚙️ Generation Parameters Form
max_new_tokens = 60 #@param {type:"slider", min:10, max:300, step:10}
temperature = 0.7 #@param {type:"slider", min:0.1, max:2.0, step:0.1}
top_k = 50 #@param {type:"slider", min:1, max:100, step:1}
top_p = 0.95 #@param {type:"slider", min:0.5, max:1.0, step:0.05}

print(f"⚙️ Parameters set: max_new_tokens={max_new_tokens}, temperature={temperature}, top_k={top_k}, top_p={top_p}")

# 💬 Step 4: Interactive Chat Interface
Run this cell to start chatting. You can write your questions, and it will compute and display the generated text along with the **tokens per second** rate.

- Type `exit`, `quit` or `q` to stop the console session.
- You can write `/config max=100 temp=0.5 topk=30` in the prompt to update parameters directly from the chat line!

In [ ]:
# ==============================================================================
# STEP 4: INTERACTIVE CHAT CONSOLE LOOP
# ==============================================================================

print("💬 BitNet Chat Console started! Type 'exit' to quit.")
print(f"⚙️ Active Config: max={max_new_tokens} tokens | temp={temperature} | top_k={top_k} | top_p={top_p}\n")

while True:
    try:
        user_input = input("👤 User: ")
        if user_input.strip().lower() in ['exit', 'quit', 'q']:
            print("👋 Session closed. Goodbye!")
            break
            
        if not user_input.strip():
            continue
            
        # Check for chat command parameters config
        if user_input.strip().startswith("/config"):
            parts = user_input.split()
            for part in parts[1:]:
                if part.startswith("max="):
                    max_new_tokens = int(part.split("=")[1])
                elif part.startswith("temp="):
                    temperature = float(part.split("=")[1])
                elif part.startswith("topk="):
                    top_k = int(part.split("=")[1])
                elif part.startswith("topp="):
                    top_p = float(part.split("=")[1])
            print(f"⚙️ Param updated: max_new_tokens={max_new_tokens}, temperature={temperature}, top_k={top_k}, top_p={top_p}")
            continue
            
        # Prepare tokens
        inputs = tokenizer(user_input, return_tensors="pt").to(device)
        
        # Token generation timing
        start_time = time.time()
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                top_k=top_k,
                top_p=top_p,
                temperature=temperature
            )
        end_time = time.time()
        
        # Decoded response
        prompt_len = inputs.input_ids.shape[1]
        response_tokens = outputs[0][prompt_len:]
        response = tokenizer.decode(response_tokens, skip_special_tokens=True)
        
        # Performance calculations
        tokens_gen = len(response_tokens)
        duration = end_time - start_time
        tokens_per_sec = tokens_gen / duration if duration > 0 else 0.0
        
        print(f"🤖 BitNet: {response.strip()}")
        print(f"   ⏱️ [Speed: {tokens_gen} tokens generated in {duration:.3f}s ({tokens_per_sec:.2f} tokens/sec)]\n")
        
    except KeyboardInterrupt:
        print("\n👋 Chat closed by user.")
        break
    except Exception as e:
        print(f"❌ Error in generation: {str(e)}\n")